# ****DATA COLLECTION****

Notebook ini mendokumentasikan sumber data, mencatat jejak berkas mentah, dan memeriksa keutuhannya sebelum dipakai untuk pembersihan dan pemodelan.

<p align="center">
    <img src="../xLain_Lain/img/dc_0.png" alt="Data Collection" witdh="100" height="500">
</p>

---

### ****DAFTAR ISI****

0. Persiapan Lingkungan Kerja dan Jejak Berkas
1. Sumber Data dan Parameter Pengunduhan
2. Pemeriksaan Struktur Berkas Mentah
3. Pemeriksaan Keutuhan dan Data Kosong
4. Kamus Data dan Keterbatasan Sumber
5. Ringkasan dan Langkah Berikutnya

### ****0. Persiapan Lingkungan Kerja dan Jejak Berkas****

#### ****0.1 Mengapa langkah ini penting?****

Data adalah fondasi seluruh hasil proyek. Agar orang lain dapat memastikan bahwa mereka memakai data yang sama persis dengan yang kami pakai, kami mencatat jejak berkas mentah: nama, ukuran, dan sidik digital SHA-256. Bila berkas berubah sedikit saja, misalnya tersimpan ulang atau teredit, nilai SHA-256 akan berbeda.

#### ****0.2 Mencatat jejak berkas****

Sel berikut memuat paket yang dibutuhkan lalu mencatat tiga hal:
1. Nama berkas dan keberadaannya di folder data mentah.
2. Ukuran berkas.
3. Nilai SHA-256 isi berkas.

In [1]:
import hashlib
import numpy as np
import pandas as pd

import config
from src.data_utils import muat_pihps_lebar

print("=" * 62)
print("Berkas raw nya")
print("=" * 62)
print(f"Nama berkas : {config.FILE_RAW.name}")
print(f"Folder : {config.FILE_RAW.parent.name}")
print(f"Ada di disk : {config.FILE_RAW.exists()}")
print(f"Ukuran berkas : {config.FILE_RAW.stat().st_size / 1024:.1f} KB")
print(f"SHA-256 : {hashlib.sha256(config.FILE_RAW.read_bytes()).hexdigest()}")

Berkas raw nya
Nama berkas : dataset_harga_cabai_rawit_merah_jatim_pihps.xlsx
Folder : Raw
Ada di disk : True
Ukuran berkas : 49.6 KB
SHA-256 : 7f2600a99d3034b0305dec5368e313b1e9395babaab22c34c98d6f549987bc64


>INTERPRETASI (Jejak berkas mentah)
>
>Berkas mentah berada di folder Raw dengan ukuran 49,6 KB. Nilai SHA-256 yang tercatat adalah 7f2600a99d3034b0305dec5368e313b1e9395babaab22c34c98d6f549987bc64. Nilai ini berfungsi sebagai sidik digital berkas: siapa pun yang memakai berkas yang sama persis akan memperoleh nilai yang sama, sehingga hasil notebook dapat dibandingkan antaranggota tim dan antarwaktu. Bila suatu saat nilai ini berubah, berarti isi berkas berubah dan seluruh hasil harus dijalankan ulang.

### ****1. Sumber Data dan Parameter Pengunduhan****

#### ****1.1 Mengapa langkah ini penting?****

Hasil peramalan hanya dapat dinilai bila asal data jelas. Karena itu parameter pengunduhan dicatat selengkap mungkin, sehingga siapa pun dapat mengunduh ulang data yang sama dan memeriksa keabsahannya.

#### ****1.2 Sumber data****

Data diambil dari Pusat Informasi Harga Pangan Strategis (PIHPS) Nasional yang dikelola Bank Indonesia, melalui halaman Tabel Harga di situs bi.go.id/hargapangan. Data harga pasar tradisional dicatat pada hari kerja.

#### ****1.3 Parameter pengunduhan****

| Parameter | Nilai |
|---|---|
| Menu | Tabel Harga, Pasar Tradisional, Berdasarkan Daerah |
| Komoditas | Cabai Rawit Merah |
| Provinsi | Jawa Timur |
| Tipe laporan | Laporan Harian |
| Tanggal mulai | 1 Januari 2020 |
| Tanggal selesai | 10 November 2026 (tanggal terakhir pada berkas) |
| Satuan harga | Rupiah per kilogram |

#### ****1.4 Isi berkas****

Berkas memuat dua baris: Cabai Rawit sebagai baris induk kategori, dan Cabai Rawit Merah sebagai komoditas yang kami teliti. Hanya baris Cabai Rawit Merah yang dipakai untuk peramalan.

### ****2. Pemeriksaan Struktur Berkas Mentah****

#### ****2.1 Mengapa langkah ini penting?****

Berkas dari sumber resmi tidak selalu berbentuk tabel yang siap dianalisis. Memeriksa bentuk aslinya terlebih dahulu mencegah kesalahan pembacaan yang sulit dilacak di tahap berikutnya, misalnya kolom yang tergeser atau tanggal yang salah terbaca.

#### ****2.2 Memeriksa struktur berkas****

Sel berikut membaca berkas apa adanya, seluruh sel sebagai teks tanpa konversi, lalu menampilkan:
1. Bentuk tabel dan jumlah kolom tanggal.
2. Nama kolom label serta kolom tanggal pertama dan terakhir.
3. Isi kedua baris untuk lima kolom tanggal pertama.

In [ ]:
mentah = pd.read_excel(config.FILE_RAW, header=0, dtype=str)

print("=" * 62)
print("Struktur berkas")
print("=" * 62)
print(f"Bentuk tabel (baris, kolom) : {mentah.shape}")
print(f"Jumlah kolom tanggal : {mentah.shape[1] - 2}")
print(f"Nama dua kolom label : {list(mentah.columns[:2])}")
print(f"Kolom tanggal pertama : {mentah.columns[2]}")
print(f"Kolom tanggal terakhir : {mentah.columns[-1]}")
print()
print("Isi baris dan lima kolom tanggal pertama:")
print(mentah.iloc[:, :7].to_string(index=False))

Struktur berkas
Bentuk tabel (baris, kolom) : (2, 1792)
Jumlah kolom tanggal : 1790
Nama dua kolom label : ['No', 'Komoditas (Rp)']
Kolom tanggal pertama : 01/ 01/ 2020
Kolom tanggal terakhir : 10/ 11/ 2026

Isi baris dan lima kolom tanggal pertama:
No    Komoditas (Rp) 01/ 01/ 2020 02/ 01/ 2020 03/ 01/ 2020 06/ 01/ 2020 07/ 01/ 2020
 I       Cabai Rawit            -       28,950       30,150       36,350       37,250
 1 Cabai Rawit Merah            -       34,550       36,850       47,200       47,550


>INTERPRETASI (Struktur berkas)
>
>Berkas berbentuk tabel lebar berukuran 2 baris dan 1.792 kolom, yaitu dua kolom label (No dan Komoditas) ditambah 1.790 kolom tanggal dari 01/ 01/ 2020 sampai 10/ 11/ 2026. Kedua baris berisi Cabai Rawit (baris induk bernomor I) dan Cabai Rawit Merah (bernomor 1, komoditas yang diteliti). Nilai kedua baris berbeda, sehingga keduanya tidak boleh tertukar. Karena itu fungsi pemuat data mengambil baris berdasarkan nama komoditas, bukan berdasarkan urutan.
>
>Tiga hal perlu dibersihkan saat konversi: tanggal tertulis dengan spasi setelah garis miring dan berurutan hari, bulan, tahun; harga tersimpan sebagai teks dengan koma sebagai pemisah ribuan; dan data kosong ditandai dengan tanda hubung.
>
>Contoh lima kolom pertama menunjukkan dua hal. Pertama, 1 Januari 2020 kosong (hari libur). Kedua, tanggal melompat dari 03/ 01/ 2020 ke 06/ 01/ 2020 karena akhir pekan dilewati, sehingga pencatatan memang hanya pada hari kerja. Harga Cabai Rawit Merah naik dari Rp 36.850 pada 3 Januari menjadi Rp 47.200 pada 6 Januari 2020, kenaikan sekitar 28% yang terjadi setelah akhir pekan. Ini sejalan dengan temuan di Notebook 1 bahwa sebagian lonjakan harian terbesar jatuh pada hari Senin.

### ****3. Pemeriksaan Keutuhan dan Data Kosong****

#### ****3.1 Mengapa langkah ini penting?****

Model peramalan deret waktu sangat bergantung pada urutan dan kelengkapan waktu. Tanggal yang kembar, tidak berurutan, atau terlewat dapat menggeser hubungan antara satu hari dan hari berikutnya tanpa kami sadari. Data kosong juga harus dikenali dengan benar agar tidak tercampur dengan data yang gagal terbaca.

#### ****3.2 Yang diperiksa****

Sel berikut memeriksa delapan hal dan menandai masing-masing sebagai LULUS atau PERIKSA:
1. Tanggal tidak ada yang kembar.
2. Tanggal terurut menaik.
3. Seluruh tanggal adalah hari kerja.
4. Tidak ada hari kerja yang terlewat.
5. Jarak antar tanggal hanya satu hari atau tiga hari (melewati akhir pekan).
6. Tidak ada tanggal di masa depan.
7. Sel kosong hanya yang bertanda tanda hubung, sehingga tidak ada sel yang gagal dibaca.
8. Seluruh harga yang terbaca bernilai positif.

Setelah itu sel menampilkan rincian data kosong: jumlah per tahun, deretan kosong terpanjang, dan daftar tanggalnya.

In [5]:
harga = muat_pihps_lebar(config.FILE_RAW, config.KOMODITAS)
tgl = harga.index

def periksa(nama, lulus, rincian=""):
    status = "Lulus" if lulus else "Periksa"
    print(f"[{status}] {nama}" + (f" -> {rincian}" if rincian else ""))

print("=" * 62)
print("Pemeriksaan keutuhan data")
print("=" * 62)

periksa("Tanggal tidak ada yang kembar", tgl.is_unique)
periksa("Tanggal terurut menaik", tgl.is_monotonic_increasing)
periksa("Semua tanggal adalah hari kerja", bool((tgl.day_of_week < 5).all()))

hilang = pd.bdate_range(tgl.min(), tgl.max()).difference(tgl)
periksa("Tidak ada hari kerja yang terlewat", len(hilang) == 0, f"{len(hilang)} hari kerja hilang")

selisih = pd.Series(tgl).diff().dt.days.dropna().astype(int).value_counts().sort_index()
periksa("Jarak antar tanggal hanya 1 hari atau 3 hari (akhir pekan)",
        set(selisih.index) <= {1, 3}, str(selisih.to_dict()))

periksa("Tidak ada tanggal di masa depan", tgl.max() <= pd.Timestamp.today().normalize())

baris_mentah = mentah[mentah.iloc[:, 1].str.strip() == config.KOMODITAS].iloc[0, 2:]
jumlah_strip = int((baris_mentah.str.strip() == "-").sum())
periksa("Sel kosong hanya yang bertanda '-' (tidak ada sel gagal dibaca)",
        jumlah_strip == int(harga.isna().sum()),
        f"{jumlah_strip} bertanda '-', {int(harga.isna().sum())} kosong setelah dibaca")

periksa("Semua harga yang terbaca bernilai positif", bool((harga.dropna() > 0).all()))

print()
print("=" * 62)
print("Data Kosong")
print("=" * 62)
kosong = harga.isna()
print(f"Jumlah hari kosong: {int(kosong.sum())} dari {len(harga)} ({kosong.mean() * 100:.1f}%)")
print()
print("Per tahun:")
print(kosong.groupby(tgl.year).sum().astype(int).to_string())

terpanjang, mulai, jumlah, awal = 0, None, 0, None
for t, k in kosong.items():
    if k:
        if jumlah == 0:
            awal = t
        jumlah += 1
        if jumlah > terpanjang:
            terpanjang, mulai = jumlah, awal
    else:
        jumlah = 0
akhir = tgl[tgl.get_loc(mulai) + terpanjang - 1]
print()
print(f"Deretan kosong terpanjang: {terpanjang} hari kerja berturut-turut "
      f"({mulai.date()} sampai {akhir.date()})")
print()
print("Daftar tanggal kosong:")
for tahun, kelompok in harga[kosong].groupby(harga[kosong].index.year):
    print(f"  {tahun}: " + ", ".join(d.strftime("%d-%m") for d in kelompok.index))

Pemeriksaan keutuhan data
[Lulus] Tanggal tidak ada yang kembar
[Lulus] Tanggal terurut menaik
[Lulus] Semua tanggal adalah hari kerja
[Lulus] Tidak ada hari kerja yang terlewat -> 0 hari kerja hilang
[Lulus] Jarak antar tanggal hanya 1 hari atau 3 hari (akhir pekan) -> {1: 1431, 3: 358}
[Lulus] Tidak ada tanggal di masa depan
[Lulus] Sel kosong hanya yang bertanda '-' (tidak ada sel gagal dibaca) -> 44 bertanda '-', 44 kosong setelah dibaca
[Lulus] Semua harga yang terbaca bernilai positif

Data Kosong
Jumlah hari kosong: 44 dari 1790 (2.5%)

Per tahun:
tanggal
2020    17
2021    13
2022    14
2023     0
2024     0
2025     0
2026     0

Deretan kosong terpanjang: 6 hari kerja berturut-turut (2022-04-29 sampai 2022-05-06)

Daftar tanggal kosong:
  2020: 01-01, 25-03, 10-04, 01-05, 07-05, 21-05, 25-05, 31-07, 17-08, 20-08, 28-10, 29-10, 30-10, 09-12, 24-12, 25-12, 31-12
  2021: 01-01, 12-02, 11-03, 02-04, 12-05, 13-05, 14-05, 26-05, 01-06, 20-07, 11-08, 17-08, 20-10
  2022: 01-02, 28-0

>INTERPRETASI (Keutuhan data dan data kosong)
>
>Seluruh delapan pemeriksaan keutuhan lulus. Jarak antar tanggal hanya terdiri dari 1.431 selang satu hari dan 358 selang tiga hari. Angka 358 sama dengan jumlah akhir pekan yang dilewati, dan tidak ada jarak lain, sehingga tidak ada hari kerja yang terlewat: hari libur nasional tetap muncul sebagai kolom tanggal dan hanya nilainya yang kosong. Pemeriksaan sel kosong juga menunjukkan 44 sel bertanda tanda hubung dan 44 nilai kosong setelah dibaca, artinya tidak ada harga yang gagal terbaca lalu diam-diam menjadi kosong.
>
>Data kosong berjumlah 44 hari (2,5%): 17 hari pada 2020, 13 hari pada 2021, 14 hari pada 2022, dan tidak ada sejak 2023 sampai 2026. Tanggal-tanggal kosong tampak bertepatan dengan hari libur nasional dan cuti bersama, misalnya 1 Januari, 17 Agustus pada tiga tahun tersebut, dan deretan 29 April sampai 6 Mei 2022 di sekitar Lebaran, yang menjadi deretan kosong terpanjang (6 hari kerja berturut-turut). Kecocokan dengan kalender libur ini masih dugaan karena belum dicek satu per satu.
>
>Karena seluruh data kosong berada pada 2020 sampai 2022, cara penanganannya hanya memengaruhi bagian awal data (calon data latih) dan tidak menyentuh bagian akhir yang akan menjadi data uji. Namun deretan 6 hari kosong berturut-turut perlu penanganan hati-hati: bila diisi dengan harga hari sebelumnya, harga akan tampak diam selama enam hari padahal nilai sebenarnya tidak diketahui. Pilihan penanganannya dibahas di Notebook 3.

### ****4. Kamus Data dan Keterbatasan Sumber****

#### ****4.1 Mengapa langkah ini penting?****

Kamus data memastikan seluruh anggota tim dan pembaca memahami arti tiap kolom dengan cara yang sama. Mencatat keterbatasan sumber sejak awal juga mencegah kami menarik kesimpulan yang melampaui apa yang sebenarnya dapat dikatakan oleh data.

#### ****4.2 Kamus data****

Berkas mentah:

| Kolom | Isi | Bentuk asli |
|---|---|---|
| No | Nomor urut baris (I untuk induk kategori, 1 untuk komoditas) | Teks |
| Komoditas (Rp) | Nama komoditas | Teks |
| Kolom tanggal (1.790 kolom) | Harga pada tanggal tersebut | Teks berformat hari/ bulan/ tahun, nilai berformat ribuan dengan koma |

Data kerja setelah dibaca oleh fungsi pemuat:

| Variabel | Arti | Tipe | Satuan |
|---|---|---|---|
| tanggal | Hari kerja pencatatan harga | Tanggal | Hari |
| harga | Harga cabai rawit merah | Bilangan | Rupiah per kilogram |

Nilai harga kosong (NaN) berarti tidak ada pencatatan pada tanggal tersebut.

#### ****4.3 Keterbatasan sumber****

1. Pencatatan hanya pada hari kerja, sehingga tidak ada informasi harga untuk akhir pekan.
2. Hari libur nasional dan cuti bersama tercatat sebagai data kosong, dan deretan kosong terpanjang mencapai 6 hari kerja.
3. Berkas hanya memuat satu nilai harga per hari untuk seluruh Jawa Timur. Informasi pasar mana yang disurvei, jumlah pasar, dan sebaran harga antarpasar tidak terlihat pada berkas, sehingga tidak dapat diperiksa.
4. Cakupan pasar dan cara pelaporan dari waktu ke waktu tidak dapat dipastikan dari berkas.
5. Berkas tidak memuat keterangan kualitas atau ukuran cabai selain nama komoditas.
6. Sebagian lonjakan harga setelah akhir pekan dapat mencerminkan jeda pencatatan, bukan perubahan yang terjadi dalam satu hari.

### ****5. Ringkasan dan Langkah Berikutnya****

#### ****5.1 Ringkasan****

1. Sumber data jelas dan terdokumentasi: PIHPS Nasional, pasar tradisional, Jawa Timur, laporan harian, 1 Januari 2020 sampai 10 November 2026.
2. Berkas mentah tercatat dengan sidik digital SHA-256 sehingga keasliannya dapat diperiksa.
3. Struktur berkas berbentuk tabel lebar dengan 1.790 kolom tanggal, dan baris komoditas yang dipakai adalah Cabai Rawit Merah.
4. Seluruh pemeriksaan keutuhan lulus: tanggal unik, terurut, hanya hari kerja, dan tidak ada hari kerja yang terlewat.
5. Terdapat 44 hari kosong (2,5%) yang seluruhnya berada pada 2020 sampai 2022, dengan deretan terpanjang 6 hari kerja.

#### ****5.2 Langkah berikutnya****

Pada Notebook 3 (pra-pemrosesan dan eksplorasi) kami akan:
1. Memutuskan cara menangani 44 hari kosong, terutama deretan 6 hari pada 2022.
2. Memeriksa nilai yang tidak wajar, termasuk harga terendah Rp 10.250 pada 21 Agustus 2020 dan lonjakan harian terbesar.
3. Mengubah data ke format panjang dan menyimpannya ke folder Dataset/Clean.
4. Menetapkan pembagian data latih, validasi, dan uji secara berurutan waktu.
5. Menyelidiki pola harga untuk menentukan rentang semesta dan jumlah interval awal.